In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.pyplot as plt

## Read training and test datasets

In [ ]:
# helper function for reading datatset
def read_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d') # convert it to datatime

    return df

In [ ]:
df_train = read_data('data/kospi_train.csv')
df_test = read_data('data/kospi_test.csv')

len(df_train), len(df_test)

In [ ]:
# the training dataset has daily KOSPI index from 2019 to 2022
df_train

In [ ]:
# the test dataset has daily KOSPI index in 2023
df_test

## Part 1. Train regression models to predict the next day's `close` using `Open`, `Low`, `High`, `Close`, `Volume` of previous days as predictors using *only* df_train. Cross-validate to select the best model. Evaluate the accuracy of your model using `df_test`.


In [ ]:
# GOOD LUCK

# Assume df_train and df_test are preloaded
def create_features(df):
    # Shift features to use previous day's data to predict next day's Close
    df_feat = df[['Open', 'High', 'Low', 'Close', 'Volume']].shift(-1)
    df_feat['Target'] = df['Close']  # today's Close is the target
    df_feat = df_feat.dropna()
    return df_feat

# Prepare features
train_feat = create_features(df_train)
test_feat = create_features(df_test)

X_train = train_feat.drop(columns='Target')
y_train = train_feat['Target']
X_test = test_feat.drop(columns='Target')
y_test = test_feat['Target']

# Standardize features (optional but useful for some models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(),
    'SVR': SVR(),
    'GradientBoosting': GradientBoostingRegressor(),
}

best_score = float('inf')
best_model = None
best_name = ""
tscv = TimeSeriesSplit(n_splits=5)

# Store scaled features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cross-validation to find the best model
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=tscv, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    avg_rmse = rmse_scores.mean()
    print(f"{name} CV RMSE: {avg_rmse:.4f}")
    
    if avg_rmse < best_score:
        best_score = avg_rmse
        best_model = model
        best_name = name

print(f"\nSelected best model: {best_name} with CV RMSE: {best_score:.4f}")

# Train best model on full training set
best_model.fit(X_train_scaled, y_train)
preds = best_model.predict(X_test_scaled)

# Evaluate on test set
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, preds)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²: {r2:.4f}")


## Part 2. Extend the regression model by adding some extra features of your choice. You can use any statistics publicly available. 

In [ ]:
# GOOD LUCK

plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values, label='Actual Close', color='blue')
plt.plot(y_test.index, preds, label='Predicted Close', color='orange', linestyle='--')

plt.title(f'{best_name} - Actual vs Predicted Close Prices')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.6, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # Diagonal line

plt.title(f'{best_name} - Scatter Plot of Actual vs Predicted Close Prices')
plt.xlabel('Actual Close Price')
plt.ylabel('Predicted Close Price')
plt.grid(True)
plt.tight_layout()
plt.show()

# 2nd Project


In [ ]:
def read_other_data(file_path):
    df = pd.read_csv(file_path)

    return df

In [ ]:
df_train = read_data('data/train.csv')

df_train